In [2]:
%matplotlib widget

In [3]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
from g4beam import *
from scan import *
from scipy.optimize import differential_evolution
from matplotlib.cm import viridis
import math
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import re
import sys
from pathlib import Path
from matplotlib import cm
import numpy as np
import pandas as pd
from tqdm import *
import pickle
import itertools
import os
from tabulate import tabulate
import tempfile
import glob
import json
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

### Combining Multiple Elements of the Dispersion Suppressor

In [4]:
# If particles_after isn't updated with Z = 0.
def convertZ(input_file, output_file):
    event_id_counter = 1
    with open(input_file, "r") as infile, open(output_file, "w") as outfile:
        for line in infile:
            # Skip header lines (those starting with #)
            if line.strip().startswith("#"):
                outfile.write(line)
                continue

            # Split the line into columns
            parts = line.strip().split()
            if len(parts) >= 12:
                parts[2] = "0"  # Set the 3rd column (z) to 0
                # Replace event ID (assuming it's the 9th column, zero-based index 8)
                # Adjust if your event ID is in a different column
                parts[8] = str(event_id_counter)
                event_id_counter += 1
                new_line = " ".join(parts)
                outfile.write(new_line + "\n")
            else:
                # Handle lines that don't match expected format
                outfile.write(line)
    print(f"Updated file saved as '{output_file}'")
    os.remove(input_file)
    return None
#convertZ("particles_afterupt.txt", "particles_after.txt")

## Wedge

In [5]:
# Make Sure g4bl is here
import os
os.environ["PATH"] += os.pathsep + "/home/incik/G4beamline-3.08/bin"
import shutil
print(shutil.which("g4bl"))

/home/incik/G4beamline-3.08/bin/g4bl


In [6]:
# --------------- Calculation Functions -----------------
def read_g4bl_params(filename):
    """
    Reads 'param' definitions from a .g4bl file and returns a dict of parameter names and values.
    """
    params = {}
    pattern = re.compile(r"param\s+(?:-unset\s+)?(\w+)=([^\s#]+)")
    with open(filename, "r") as f:
        for line in f:
            line = line.strip()
            if not line.startswith("param"):
                continue
            m = pattern.search(line)
            if m:
                key, val = m.groups()
                try:
                    params[key] = float(eval(val, {"__builtins__": None, "pi": math.pi}))
                except Exception:
                    params[key] = val  # keep as string if not numeric
    return params

# Compute wedge elements
def compute_wedge_geometry(params):
    """
    Given parameters like absLEN3, abshgt, abswidth, abshalfangle3, compute geometry info.
    """
    W = params.get("absLEN3")
    H = params.get("abshgt")
    L = params.get("abswidth")
    half_angle_deg = params.get("abshalfangle3")
    offset = params.get("absoffset3", 0)

    if L and half_angle_deg:
        half_angle_rad = math.radians(half_angle_deg)
        L_centerline = W / math.sin(half_angle_rad)

    return {
        "Length_Wedge": L,
        "Height_Base": H,
        "Width_Base": W,
        "Half-angle": half_angle_deg,
        "Centerline_Coords": L_centerline,
        "Offset": offset
    }

# Write values to the prepared G4BL template
def write_input_from_template(template_path, out_path, replacements):
    with open(template_path, 'r') as f:
        txt = f.read()
    try:
        txt = txt.format(**replacements)
    except KeyError as e:
        raise RuntimeError(f"Template substitution failed; missing placeholder: {e}")
    with open(out_path, 'w') as f:
        f.write(txt)

#### Wedge Parameters

In [7]:
# ---------------- USER CONFIG ----------------
G4BEAMLINE_CMD = "g4bl"
TEMPLATE_FILE = "G4_FinalCooling_dispsup_Template.g4bl"
OUTPUT_DIR = "dispsup_runs"
VD_FILENAME = "vd_dispsup.txt"  # When we sweep, we change this file
N_PARTICLES = 5000                  # increase for lower noise
G4BLFILE = f"G4_FinalCooling_dispsup_run"

os.makedirs(OUTPUT_DIR, exist_ok=True)

params = read_g4bl_params(TEMPLATE_FILE)
geom = compute_wedge_geometry(params)

print("Parameters found:")
for k, v in params.items():
    print(f"  {k:15s} = {v}")

print("Derived geometry:")
for k, v in geom.items():
    print(f"  {k:20s}: {v}")

Parameters found:
  zbegin          = 0.0
  steppingFormat  = N,GLOBAL,CL,STEP,VOL,PROCESS,P,KE,POLAR,B
  fieldVoxels     = 400,400,400
  maxStep         = 0.5
  minRangeCut     = 1.0
  nparticles      = {N_PARTICLES}
  beamfile        = particles_before.txt
  pi              = 3.141592654
  degrad          = $pi/180
  abshgt          = 10.0
  abswidth        = 100.0
  absLEN3         = 18.0
  abshalfangle3   = 45.0
  absoffset3      = 3.2
  wedge_z         = 0.5*$absLEN3
  VDRad           = 60.0
  wedgeAxis       = 0.0
  noWedge         = 0.0
Derived geometry:
  Length_Wedge        : 100.0
  Height_Base         : 10.0
  Width_Base          : 18.0
  Half-angle          : 45.0
  Centerline_Coords   : 25.455844122715714
  Offset              : 3.2


## Placing Elements

In [13]:
# Calculate all will-be-added parameters in the template
def add_missing_params(calcparams_given):
    GAP = 0.1  # in mm (can be up to 1.0 safely)
    
    L_Q1 = float(calcparams_given["Q1_length"])
    L_D1 = float(calcparams_given["Drift1_length"])
    L_Q2 = float(calcparams_given["Q2_length"])
    L_D2 = float(calcparams_given["Drift2_length"])
    L_B1 = float(calcparams_given["B1_length"])

    wedge_end = geom["Centerline_Coords"]
    Q1_z      = geom["Centerline_Coords"] + (L_Q1/2) + GAP
    Drift1_z = Q1_z + (L_Q1/2) + (L_D1/2) + GAP
    Q2_z     = Drift1_z + (L_D1/2) + (L_Q2/2) + GAP
    Drift2_z = Q2_z + (L_Q2/2) + (L_D2/2)+ GAP
    B1_z     = Drift2_z + (L_D2/2) + (L_B1/2) + GAP
    VD_z     = B1_z + (L_B1/2) + 10.0 + GAP

    Q1_end = Q1_z + (L_Q1/2)
    D1_end = Drift1_z + (L_D1/2)
    Q2_end = Q2_z + (L_Q2/2)
    D2_end = Drift2_z + (L_D2/2)
    B1_end = B1_z + (L_B1/2)

    add_params = {"wedge_end": wedge_end, "Q1_z": Q1_z, "Drift1_z": Drift1_z, "Q2_z": Q2_z, "Drift2_z":Drift2_z, "B1_z": B1_z,"VD_z": VD_z, 
            "B1_end": B1_end, "Q2_end": Q2_end,  "D1_end": D1_end, "Q1_end": Q1_end,  "D2_end": D2_end}

    calcparams_given.update(add_params)
    
    for k, v in calcparams_given.items():
        if k == "N_PARTICLES" or k == "VD_FILENAME":
            continue
        else:
            calcparams_given[k] = float(v)
    
    return calcparams_given

## Dave's Design

##### Plugging in a single Quadrupole and Sweeping

In [14]:
# Helper Function for finding where the zero value D is reached if it is
def find_first_zero_crossing(x_vals, y_vals):
    x_vals = np.array(x_vals)
    y_vals = np.array(y_vals)
    for i in range(len(y_vals)-1):
        if y_vals[i] * y_vals[i+1] <= 0:
            x0, x1 = x_vals[i], x_vals[i+1]
            y0, y1 = y_vals[i], y_vals[i+1]
            if y1 != y0:
                return x0 - y0 * (x1 - x0) / (y1 - y0)
            return x0
    return np.nan

In [16]:
# ======================================================================
# MAIN SWEEP STORAGE
# ======================================================================
quad_sweep_results = []
Q1_field_range = np.arange(-2.0, 2.0, 0.1)
Q1_length_range = np.linspace(100, 1000, 10) # in mm
Q1_thickness_range = np.linspace(100, 300, 10)
Q1_inner_radius_range = np.linspace(100, 300, 10)

# ======================================================================
# FULL QUADRUPOLE PARAMETER SCAN
# ======================================================================

for L_val in Q1_length_range:
    for thick_val in Q1_thickness_range:
        for r_in_val in Q1_inner_radius_range:

            # Prepare lists for this (L, thickness, radius) group
            Qfield_vals = []
            Dx_vals = []; Dy_vals = []; Dxp_vals = []; Dyp_vals = []
            emitx_vals = []; emity_vals = []; emitz_vals = []
            transmission_vals = []

            print(f"--- Scanning Q1_field for L={L_val:.1f} mm, t={thick_val:.1f} mm, rin={r_in_val:.1f} mm ---")

            for i, Q_val in enumerate(tqdm(Q1_field_range, desc="Q1 Field Sweep")):

                # Remove old field map
                if os.path.exists("field_cell.dat"):
                    os.remove("field_cell.dat")

                # ----------------------------------------------------------
                # Build template parameters
                # ----------------------------------------------------------
                
                var_names = ["N_PARTICLES",  "B1_field", "B1_width", "B1_height", "B1_length", 
                "Q1_gradient", "Q1_length", "radius_q", "thickness", "Q1_z",
                "Q2_gradient", "Q2_length","Drift1_width", "Drift1_height", "Drift1_length",  
                "Drift2_width", "Drift2_height","Drift2_length", "VD_FILENAME"]
                
                xvec = np.array([
                    int(N_PARTICLES), 0.2101, 122.8026, 265.3664, 83.8629,
                    Q_val, L_val, r_in_val, thick_val, 10, 
                    -0.1835, 183.4535, 173.3462, 243.1225, 139.7656, 
                    191.618, 121.9246, 86.819, f"{OUTPUT_DIR}/{VD_FILENAME}_{L_val:.1f}_{thick_val:.1f}_{r_in_val:.1f}_{i}.txt"])

                params = {k: v for k, v in zip(var_names, xvec)}
                add_missing_params(params)

                # ----------------------------------------------------------
                # Write .g4bl input file
                # ----------------------------------------------------------
                g4file = f"{OUTPUT_DIR}/{G4BLFILE}_{L_val:.1f}_{thick_val:.1f}_{r_in_val:.1f}_{i}.g4bl"

                write_input_from_template(TEMPLATE_FILE, g4file, params)

                # ----------------------------------------------------------
                # Run G4Beamline
                # ----------------------------------------------------------
                subprocess.run(["g4bl", g4file], capture_output=True, text=True, check=True)

                out_file = params["VD_FILENAME"]
                df = read_trackfile(out_file)

                with open(out_file) as f:
                    N_out = sum(1 for line in f if not line.startswith("#") and line.strip())

                trans_percent = 100.0 * N_out / N_PARTICLES
                transmission_vals.append(trans_percent)

                # Compute optical parameters
                x_params, y_params, z_emit = calc_all_params(df)

                Qfield_vals.append(Q_val)
                Dx_vals.append(x_params[4])
                Dy_vals.append(y_params[4])
                Dxp_vals.append(x_params[5])
                Dyp_vals.append(y_params[5])

                emitx_vals.append(x_params[0])
                emity_vals.append(y_params[0])
                emitz_vals.append(z_emit)

            # ===============================================================
            # STORE THIS BLOCK OF SWEEP RESULTS
            # ===============================================================
            quad_sweep_results.append({
                "Q1_length": L_val,
                "Q1_thickness": thick_val,
                "Q1_inner_radius": r_in_val,

                "Q1_field": Qfield_vals,
                "Dx": Dx_vals,
                "Dy": Dy_vals,
                "Dxp": Dxp_vals,
                "Dyp": Dyp_vals,
                "emit_x": emitx_vals,
                "emit_y": emity_vals,
                "emit_z": emitz_vals,
                "transmission": transmission_vals
            })

--- Scanning Q1_field for L=100.0 mm, t=100.0 mm, rin=100.0 mm ---


Q1 Field Sweep: 100%|██████████| 40/40 [03:27<00:00,  5.19s/it]


--- Scanning Q1_field for L=100.0 mm, t=100.0 mm, rin=122.2 mm ---


Q1 Field Sweep: 100%|██████████| 40/40 [03:27<00:00,  5.18s/it]


--- Scanning Q1_field for L=100.0 mm, t=100.0 mm, rin=144.4 mm ---


...

Q1 Field Sweep: 100%|██████████| 40/40 [05:37<00:00,  8.43s/it]


--- Scanning Q1_field for L=300.0 mm, t=255.6 mm, rin=255.6 mm ---


Q1 Field Sweep: 100%|██████████| 40/40 [05:41<00:00,  8.53s/it]


--- Scanning Q1_field for L=300.0 mm, t=255.6 mm, rin=277.8 mm ---


Q1 Field Sweep:  22%|██▎       | 9/40 [01:17<04:24,  8.54s/it]

In [ ]:
# Takes a LOT of time so save it after:
import pandas as pd

# ----------------------------------------------
# Convert quad_sweep_results into a flat table
# ----------------------------------------------
rows = []

for res in quad_sweep_results:
    L = res["Q1_length"]
    t = res["Q1_thickness"]
    r = res["Q1_inner_radius"]

    for i in range(len(res["Q1_field"])):
        rows.append({
            "Q1_length": L,
            "Q1_thickness": t,
            "Q1_inner_radius": r,
            "Q1_field": res["Q1_field"][i],

            "Dx": res["Dx"][i],
            "Dy": res["Dy"][i],
            "Dxp": res["Dxp"][i],
            "Dyp": res["Dyp"][i],

            "emit_x": res["emit_x"][i],
            "emit_y": res["emit_y"][i],
            "emit_z": res["emit_z"][i],

            "transmission": res["transmission"][i],
        })

# ----------------------------------------------
# Make DataFrame and save CSV
# ----------------------------------------------
df_all = pd.DataFrame(rows)
df_all.to_csv("quad_sweep_all_results.csv", index=False)

print("Saved: quad_sweep_all_results.csv")


### Sweep Logic:

In [ ]:
# Doing this for B1 and B1_length
B1_field_range = np.arange(-3.0, 3.0, 0.2)
B1_length_range = np.linspace(10, 100, 10) # in mm

# Initialize lists
results = []

for L_val in B1_length_range:
    Dip1 = []
    for i, B_val in enumerate(tqdm(B1_field_range, desc=f"L = {L_val:.1f}")):
        if os.path.exists("field_cell.dat"):
            os.remove("field_cell.dat")

        var_names = ["N_PARTICLES", "B1_field", "B1_width", "B1_height", "B1_length",
                    "Q1_gradient", "Q1_length", "radius_q", "B2_field", "B2_width", "B2_height", "B2_length",
                    "Drift1_width", "Drift1_height", "Drift1_length", "Drift2_width", "Drift2_height", "Drift2_length", "VD_FILENAME", "VD_AFILENAME"]
        #74.7401
        xvec = np.array([
            int(N_PARTICLES), B_val, 30.0, geom["Length_Wedge"], L_val,
            -43.4531, 73.1234, geom["Length_Wedge"], -0.0126, 167.5582, 144.8055, 53.6328,
            120.1356, geom["Length_Wedge"], 463.8105, 102.022, geom["Length_Wedge"], 316.7456,
            "vd_B1_end_achromat.txt", f"./runs/newruns/{VD_AFILENAME}_{L_val:.1f}_{i}.txt"
        ])

        calcparams = {name: val for name, val in zip(var_names, xvec)}
        add_missing_params(calcparams)
        
        write_input_from_template(
            TEMPLATE_FILE,
            f"/home/incik/Cooling_4D/AchromatOneByOne/runs/newruns/{G4BLFILE}_{L_val:.1f}_{i}.g4bl",
            calcparams
        )

        subprocess.run(["g4bl", f"/home/incik/Cooling_4D/AchromatOneByOne/runs/newruns/{G4BLFILE}_{L_val:.1f}_{i}.g4bl"], 
                    capture_output=True, text=True, check=True)

        G4BLOUTPUT = f"/home/incik/Cooling_4D/AchromatOneByOne/runs/newruns/{VD_AFILENAME}_{L_val:.1f}_{i}.txt"
        df = read_trackfile(G4BLOUTPUT)
        
        
        Dip1.append(B_val)
        
    # store all results for this L_val
    results.append({
        "B1_length": L_val,
        "Dip1": Dip1,
    })
    
    print(results)



    """# Skip if radii exceed Larmor-defined aperture
        mask, r_L_mm, delta_r_mm = check_dipole_clearance(df, B_val, L_val, calcparams["B1_width"]/2)
        
        # If less than 95% of particles clear the aperture, skip this configuration
        if np.mean(mask) < 0.95:
            print(f"Skipping B={B_val:.2f}T, L={L_val:.1f}mm: only {np.mean(mask)*100:.1f}% clear.")
            continue
    """